# 04 — Consolidação da shortlist e transição de coorte

## Objetivo

Consolidar os resultados produzidos pelo Notebook 03 e preparar a troca da coorte provisória pela definitiva sem repetir tuning, abrir o teste ou duplicar regras de carregamento.

Este notebook:

- lê a coorte por `src.data.cohort_source`;
- usa `SRAG_COHORT_KIND=provisional` como padrão explícito;
- carrega os artefatos mais recentes do Notebook 03;
- confirma a shortlist provisória;
- resume estabilidade temporal e calibração bruta;
- calcula a prontidão para substituir a fonte pela coorte definitiva;
- não treina modelos e não consulta a partição de teste.

> A troca futura será feita definindo `SRAG_COHORT_KIND=final` e rerodando os notebooks dependentes. Nenhum nome de arquivo precisa ser alterado nas células seguintes.

## 1. Bootstrap e contrato de execução

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src").is_dir() and (pasta / "configs").is_dir():
            return pasta
    raise FileNotFoundError("Raiz do projeto nao encontrada.")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import config as C


COHORT_KINDS_VALIDOS = {"provisional", "final"}
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").strip().lower()
if COHORT_KIND not in COHORT_KINDS_VALIDOS:
    raise ValueError(
        f"SRAG_COHORT_KIND invalido: {COHORT_KIND!r}. "
        f"Valores permitidos: {sorted(COHORT_KINDS_VALIDOS)}."
    )

STATUS_EXECUCAO = {"provisional": "provisoria", "final": "definitiva"}[COHORT_KIND]
TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Validação estrita do YAML

A validação abaixo impede continuar quando gates essenciais aparecem fora do bloco `gates` ou quando o YAML contém chaves duplicadas. Isso protege o handoff entre a coorte provisória e a definitiva.

In [ ]:
import yaml


class UniqueKeyLoader(yaml.SafeLoader):
    pass


def construct_mapping_unique(loader, node, deep=False):
    mapping = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise ValueError(f"Chave YAML duplicada: {key!r}")
        mapping[key] = loader.construct_object(value_node, deep=deep)
    return mapping


UniqueKeyLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    construct_mapping_unique,
)

YAML_PATH = ROOT_DIR / "configs" / "supervised.yaml"
CFG_STRICT = yaml.load(YAML_PATH.read_text(encoding="utf-8"), Loader=UniqueKeyLoader)

GATES_ESPERADOS = {
    "G1_definicao_alvo_features",
    "G2_janela_temporal",
    "G4_baseline",
    "G5_pilotos",
    "G6_finalistas_tuning",
    "G7_calibracao",
    "G8_limiar",
    "G9_holdout",
}

gates = CFG_STRICT.get("gates") or {}
gates_faltantes = sorted(GATES_ESPERADOS - set(gates))
gates_soltos = sorted(GATES_ESPERADOS & set(CFG_STRICT))

features_final_val = (CFG_STRICT.get("features") or {}).get("final")
features_final_ok = (
    not features_final_val
    if COHORT_KIND == "provisional"
    else bool(features_final_val)
)

validacao_yaml = pd.DataFrame(
    {
        "verificacao": [
            "bloco_gates_existe",
            "gates_esperados_presentes",
            "nenhum_gate_solto_na_raiz",
            "shortlist_sem_teste",
            "features_finais_consistentes_com_kind",
        ],
        "resultado": [
            isinstance(gates, dict),
            not gates_faltantes,
            not gates_soltos,
            (CFG_STRICT.get("selecao_modelos") or {}).get("teste_consultado") is False,
            features_final_ok,
        ],
    }
)

display(validacao_yaml)

falhas_yaml = validacao_yaml.loc[~validacao_yaml["resultado"], "verificacao"].tolist()
if falhas_yaml:
    raise RuntimeError(
        "O supervised.yaml precisa ser corrigido antes da consolidacao: "
        f"{falhas_yaml}"
    )

## 3. Carregamento desacoplado da coorte

A célula que diferencia provisória e definitiva está fora do notebook, em `src/data/cohort_source.py`. O restante do fluxo recebe apenas `df_coorte` e `fonte_coorte`.

In [ ]:
from src.data.supervised_artifact import load_supervised_artifact

df_coorte, manifesto_supervisionado = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)

fonte_coorte = manifesto_supervisionado["source"]
fonte_coorte_kind = manifesto_supervisionado["cohort_kind"]

print("Dataset:", manifesto_supervisionado["dataset_path"])
print("SHA-256:", manifesto_supervisionado["dataset_sha256"])

## 4. Contrato da fonte e prontidão de substituição

In [ ]:
features_cfg = CFG_STRICT.get("features") or {}
features_modelo = list(
    features_cfg.get("final")
    or (
        list(features_cfg.get("numericas") or [])
        + list(features_cfg.get("categoricas") or [])
    )
)
proibidas = set(features_cfg.get("proibidas") or [])

coluna_particao = str(CFG_STRICT["temporal"]["coluna_particao"])
coluna_alvo = str(CFG_STRICT["target"]["coluna_modelagem"])
coluna_ano = str(CFG_STRICT["periodo"]["cobertura_fonte"]["coluna_ano"])

faltantes_modelo = sorted(set(features_modelo) - set(df_coorte.columns))
vazamento_config = sorted(set(features_modelo) & proibidas)

particoes_observadas = (
    sorted(
        df_coorte[coluna_particao]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    if coluna_particao in df_coorte.columns
    else []
)

caminho_dataset_supervisionado = Path(manifesto_supervisionado["dataset_path"])

teste_fechado_manifesto = manifesto_supervisionado.get("test_opened") is False
teste_fechado_yaml = (
    (CFG_STRICT.get("selecao_modelos") or {}).get("teste_consultado") is False
)

contrato_fonte = pd.DataFrame(
    {
        "item": [
            "artefato_supervisionado_existe",
            "tipo_solicitado_corresponde",
            "features_presentes",
            "features_sem_proibidas",
            "alvo_modelagem_presente",
            "ano_presente",
            "particao_presente",
            "particoes_validas",
            "teste_fechado_no_manifesto",
            "teste_nao_consultado_no_yaml",
        ],
        "resultado": [
            caminho_dataset_supervisionado.is_file(),
            fonte_coorte_kind == COHORT_KIND,
            not faltantes_modelo,
            not vazamento_config,
            coluna_alvo in df_coorte.columns,
            coluna_ano in df_coorte.columns,
            coluna_particao in df_coorte.columns,
            set(particoes_observadas) == {"treino", "validacao", "teste"},
            teste_fechado_manifesto,
            teste_fechado_yaml,
        ],
    }
)

display(contrato_fonte)

falhas_criticas = contrato_fonte.loc[
    ~contrato_fonte["resultado"], "item"
].tolist()
if falhas_criticas:
    raise RuntimeError(
        "Falhas no contrato do dataset supervisionado: "
        f"{falhas_criticas}"
    )

## 5. Localização dos artefatos mais recentes do Notebook 03

Os resultados são descobertos por prefixo e data de modificação. Nenhum número é copiado manualmente para este notebook.

In [ ]:
def manifesto_mais_recente(
    pasta: Path,
    inicio_nome: str,
    final_nome: str,
) -> Path:
    candidatos = [
        caminho
        for caminho in pasta.iterdir()
        if caminho.is_file()
        and caminho.name.startswith(inicio_nome)
        and caminho.name.endswith(final_nome)
    ]
    if not candidatos:
        raise FileNotFoundError(
            f"Nenhum manifesto em {pasta} com inicio "
            f"{inicio_nome!r} e final {final_nome!r}."
        )
    return max(candidatos, key=lambda caminho: caminho.stat().st_mtime_ns)


inicio_manifesto = "03_cinco_familias_" + STATUS_EXECUCAO + "_"

PATH_MANIFESTO = manifesto_mais_recente(
    TABLES_DIR,
    inicio_nome=inicio_manifesto,
    final_nome="_manifesto.json",
)

manifesto_03 = json.loads(PATH_MANIFESTO.read_text(encoding="utf-8"))

if manifesto_03.get("status_coorte") != STATUS_EXECUCAO:
    raise RuntimeError(
        "O manifesto mais recente nao corresponde ao status da execucao. "
        f"Esperado: {STATUS_EXECUCAO!r}. "
        f"Encontrado: {manifesto_03.get('status_coorte')!r}."
    )

artefatos_manifesto = manifesto_03.get("artefatos") or {}

CHAVES_ARTEFATOS_OBRIGATORIAS = {
    "metricas",
    "metricas_por_ano",
    "variacao_temporal",
    "predicoes_validacao",
}

chaves_faltantes = sorted(
    CHAVES_ARTEFATOS_OBRIGATORIAS.difference(artefatos_manifesto)
)
if chaves_faltantes:
    raise KeyError(
        "O manifesto do Notebook 03 nao contem todos os caminhos "
        f"obrigatorios. Chaves ausentes: {chaves_faltantes}"
    )

PATH_METRICAS = Path(artefatos_manifesto["metricas"])
PATH_METRICAS_ANO = Path(artefatos_manifesto["metricas_por_ano"])
PATH_VARIACAO = Path(artefatos_manifesto["variacao_temporal"])
PATH_PREDICOES = Path(artefatos_manifesto["predicoes_validacao"])

caminhos_obrigatorios = {
    "metricas": PATH_METRICAS,
    "metricas_ano": PATH_METRICAS_ANO,
    "variacao": PATH_VARIACAO,
    "manifesto": PATH_MANIFESTO,
    "predicoes": PATH_PREDICOES,
}
caminhos_inexistentes = {
    nome: str(caminho)
    for nome, caminho in caminhos_obrigatorios.items()
    if not caminho.is_file()
}
if caminhos_inexistentes:
    raise FileNotFoundError(
        "Artefatos declarados no manifesto ausentes: "
        f"{caminhos_inexistentes}"
    )

metricas = pd.read_csv(PATH_METRICAS, encoding="utf-8-sig")
metricas_ano = pd.read_csv(PATH_METRICAS_ANO, encoding="utf-8-sig")
variacao = pd.read_csv(PATH_VARIACAO, encoding="utf-8-sig")
predicoes = pd.read_parquet(
    PATH_PREDICOES,
    columns=["modelo", "y_true", "y_prob"],
)

display(
    pd.DataFrame(
        {
            "artefato": list(caminhos_obrigatorios.keys()),
            "caminho": list(caminhos_obrigatorios.values()),
        }
    )
)

## 6. Consolidação da shortlist provisória

In [ ]:
selecao_cfg = CFG_STRICT.get("selecao_modelos") or {}

shortlist_yaml = list(selecao_cfg.get("finalistas") or [])
nomes_execucao = dict(selecao_cfg.get("nomes_execucao") or {})

mapeamentos_faltantes = sorted(set(shortlist_yaml).difference(nomes_execucao))
if mapeamentos_faltantes:
    raise KeyError(
        "Faltam nomes de execucao no supervised.yaml para: "
        f"{mapeamentos_faltantes}"
    )

shortlist_execucao = [nomes_execucao[nome] for nome in shortlist_yaml]

metricas_shortlist = (
    metricas.loc[metricas["modelo"].isin(shortlist_execucao)]
    .copy()
    .sort_values(["pr_auc", "brier"], ascending=[False, True])
    .reset_index(drop=True)
)

modelos_encontrados = set(metricas_shortlist["modelo"].astype(str).tolist())
if modelos_encontrados != set(shortlist_execucao):
    raise RuntimeError(
        "A shortlist do YAML nao coincide com os modelos dos artefatos. "
        f"Esperados: {shortlist_execucao}. "
        f"Encontrados: {sorted(modelos_encontrados)}."
    )

colunas_exibir = [
    coluna
    for coluna in [
        "modelo",
        "pr_auc",
        "roc_auc",
        "brier",
        "recall",
        "precision",
        "f1",
        "duracao_fit_s",
        "n_treino",
        "n_validacao",
    ]
    if coluna in metricas_shortlist.columns
]
display(metricas_shortlist[colunas_exibir])

if manifesto_03.get("teste_aberto") is not False:
    raise RuntimeError(
        "O manifesto do Notebook 03 nao confirma teste fechado."
    )

## 7. Estabilidade temporal e calibração bruta

A análise permanece descritiva. Não escolhe calibrador nem limiar.

In [ ]:
temporal_shortlist = (
    metricas_ano.loc[metricas_ano["modelo"].isin(shortlist_execucao)]
    .copy()
    .sort_values(["modelo", "ano_internacao"])
)
variacao_shortlist = (
    variacao.loc[variacao["modelo"].isin(shortlist_execucao)]
    .copy()
    .sort_values(
        ["amplitude_pr_auc", "pr_auc_media"],
        ascending=[True, False],
    )
)

resumo_calibracao_bruta = (
    predicoes.loc[predicoes["modelo"].isin(shortlist_execucao)]
    .groupby("modelo", observed=True)
    .agg(
        n=("y_true", "size"),
        prevalencia=("y_true", "mean"),
        probabilidade_media=("y_prob", "mean"),
    )
    .reset_index()
)
resumo_calibracao_bruta["diferenca_media"] = (
    resumo_calibracao_bruta["probabilidade_media"]
    - resumo_calibracao_bruta["prevalencia"]
)

display(variacao_shortlist)
display(resumo_calibracao_bruta)
display(temporal_shortlist)

## 8. Matriz de transição para a coorte definitiva

A matriz separa o que já pode permanecer igual do que precisa ser validado na chegada da coorte definitiva.

In [ ]:
features_final = (CFG_STRICT.get("features") or {}).get("final")
gate_g1 = (gates.get("G1_definicao_alvo_features") or {}).get("status")
gate_g6 = (gates.get("G6_finalistas_tuning") or {}).get("status")

particao_persistida = set(particoes_observadas) == {"treino", "validacao", "teste"}

matriz_transicao = pd.DataFrame(
    [
        {
            "componente": "Loader da coorte",
            "estado_atual": fonte_coorte_kind,
            "condicao_para_final": (
                "SRAG_COHORT_KIND=final e coorte_final.parquet disponivel"
            ),
            "troca_de_codigo": "nenhuma",
            "pronto_agora": True,
            "pronto_para_final": fonte_coorte_kind == "final",
        },
        {
            "componente": "Schema de features",
            "estado_atual": "piloto" if features_final is None else "congelado",
            "condicao_para_final": (
                "features.final preenchida apos validacao do schema"
            ),
            "troca_de_codigo": "nenhuma; somente YAML",
            "pronto_agora": not faltantes_modelo,
            "pronto_para_final": bool(features_final),
        },
        {
            "componente": "Definicao de alvo e features",
            "estado_atual": str(gate_g1),
            "condicao_para_final": "G1_definicao_alvo_features aprovado",
            "troca_de_codigo": "nenhuma; somente gate",
            "pronto_agora": True,
            "pronto_para_final": gate_g1 == "aprovado",
        },
        {
            "componente": "Particao fixa",
            "estado_atual": ", ".join(particoes_observadas) or "ausente",
            "condicao_para_final": (
                "treino, validacao e teste persistidos no dataset"
            ),
            "troca_de_codigo": "nenhuma",
            "pronto_agora": particao_persistida,
            "pronto_para_final": (
                particao_persistida and fonte_coorte_kind == "final"
            ),
        },
        {
            "componente": "Pilotos comparaveis",
            "estado_atual": "concluidos",
            "condicao_para_final": "rerodar Notebook 03 com a fonte final",
            "troca_de_codigo": "nenhuma",
            "pronto_agora": True,
            "pronto_para_final": fonte_coorte_kind == "final",
        },
        {
            "componente": "Tuning dos finalistas",
            "estado_atual": str(gate_g6),
            "condicao_para_final": (
                "confirmar tuning apos G1 e repeticao dos pilotos"
            ),
            "troca_de_codigo": "Notebook 05",
            "pronto_agora": True,
            "pronto_para_final": gate_g6 == "aprovado",
        },
    ]
)

display(matriz_transicao)

## 9. Persistência do handoff da Pessoa B

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo = f"04_handoff_{fonte_coorte_kind}_{timestamp}"

PATH_RESUMO = TABLES_DIR / f"{prefixo}_shortlist.csv"
PATH_TEMPORAL = TABLES_DIR / f"{prefixo}_temporal.csv"
PATH_CALIBRACAO = TABLES_DIR / f"{prefixo}_calibracao_bruta.csv"
PATH_TRANSICAO = TABLES_DIR / f"{prefixo}_transicao.csv"
PATH_HANDOFF = TABLES_DIR / f"{prefixo}_manifesto.json"

metricas_shortlist.to_csv(PATH_RESUMO, index=False, encoding="utf-8-sig")
temporal_shortlist.to_csv(PATH_TEMPORAL, index=False, encoding="utf-8-sig")
resumo_calibracao_bruta.to_csv(PATH_CALIBRACAO, index=False, encoding="utf-8-sig")
matriz_transicao.to_csv(PATH_TRANSICAO, index=False, encoding="utf-8-sig")

handoff = {
    "tipo_coorte": fonte_coorte_kind,
    "fonte_coorte": dict(fonte_coorte),
    "dataset_supervisionado": {
        "path": manifesto_supervisionado["dataset_path"],
        "sha256": manifesto_supervisionado["dataset_sha256"],
        "rows": manifesto_supervisionado["rows"],
        "columns": manifesto_supervisionado["columns"],
        "partition_column": manifesto_supervisionado["partition_column"],
        "partition_counts": manifesto_supervisionado["partition_counts"],
        "seed": manifesto_supervisionado["seed"],
        "test_opened": manifesto_supervisionado["test_opened"],
    },
    "status_execucao": STATUS_EXECUCAO,
    "shortlist_yaml": shortlist_yaml,
    "shortlist_execucao": shortlist_execucao,
    "teste_consultado": False,
    "tuning_realizado": False,
    "calibracao_escolhida": False,
    "limiar_escolhido": False,
    "gate_g1": gate_g1,
    "gate_g6": gate_g6,
    "features_final_congelada": bool(features_final),
    "artefatos_origem_notebook03": {
        "metricas": str(PATH_METRICAS),
        "metricas_por_ano": str(PATH_METRICAS_ANO),
        "variacao_temporal": str(PATH_VARIACAO),
        "manifesto": str(PATH_MANIFESTO),
        "predicoes_validacao": str(PATH_PREDICOES),
    },
    "artefatos_handoff": {
        "shortlist": str(PATH_RESUMO),
        "temporal": str(PATH_TEMPORAL),
        "calibracao_bruta": str(PATH_CALIBRACAO),
        "transicao": str(PATH_TRANSICAO),
    },
}

PATH_HANDOFF.write_text(
    json.dumps(handoff, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Handoff gravado:", PATH_HANDOFF)

## 10. Checklist final

O Notebook 04 está concluído quando:

- o YAML passa na validação estrita;
- a fonte da coorte é explícita;
- a shortlist do YAML coincide com os artefatos do Notebook 03;
- o manifesto confirma teste fechado;
- estabilidade temporal e calibração bruta são consolidadas;
- a matriz de transição deixa claro o que falta para a coorte definitiva;
- os artefatos de handoff são salvos.

### Próximo gate

Com a coorte provisória, a Pessoa B encerra a consolidação e prepara texto, limitações e figuras. O tuning não deve avançar como decisão final enquanto `features.final` permanecer vazio e `G1_definicao_alvo_features` estiver pendente.

Quando a coorte definitiva chegar:

1. definir `SRAG_COHORT_KIND=final`;
2. validar e preencher `features.final` no YAML;
3. aprovar G1;
4. rerodar o Notebook 03 sem alterar suas regras;
5. rerodar este Notebook 04;
6. somente então iniciar tuning limitado dos dois finalistas.